# Outstanding tasks

| Owner | Task | What needs to be done |
|---|---|---|
| C, Nico | #5 Other windows | Run the event study on [-1, +1] and [-5, +5] as well as the [-10, +10] baseline. |
| C, Nico | Significance tests | Test whether CARs differ from zero, for negative and positive events: Boehmer et al. t-statistic, Cowan's sign test and a t-test clustered by date. |
| C, Nico | Robustness | Fama-French CARs, other overlap gaps, a β = 1 model, dropping stale-price banks, and dropping events with another campaign inside the window. |
| Alissa | #6 Do NGO campaigns matter? | In `Task6.ipynb`: split CARs by COP21, event type, prominence and sentiment, run one regression as the check on the splits, and plot the main figure. |
| A, Cecilia | #3 NLP sentiment | Score each campaign from `report` in `campaigns_banks.csv`, and test whether the source links still work. Merge the score back to bank-days through `uids`. |
| D | #1 or #7 Add-on | Build one variable per campaign (media coverage or political preferences), keyed on `uid_archive`. It becomes one more split in #6. |
| D | Slides | Write the narrative and the four slides, once #6 has results. |

In [ ]:
import zipfile, io
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import re
import html

In [ ]:
DATA = Path("data") if Path("data").exists() else Path(".")   # data/ folder if there is one, otherwise the project folder
OUT = DATA / "handoff"                                        # files passed between A, B, C and D
OUT.mkdir(exist_ok=True)

if not (DATA / "sigwatch_data").exists():
    for z in ["ngo_data_bocconi.zip", "banks_data_bocconi.zip"]:
        zipfile.ZipFile(DATA / z).extractall(DATA)

---
# Part A: NGO data (owner: A, Cecilia)
Load and append the Sigwatch years, filter to US/UK/EU banks, describe the campaigns, and build the name→ISIN lookup.

**Handoff to C and D:** `handoff/events_banks.csv`, `handoff/campaigns_banks.csv`, `handoff/lookup_banks.csv`.

## A1. Load and append the Sigwatch years

In [ ]:
# Sigwatch
files = sorted((DATA / "sigwatch_data").glob("*.dta"))

sig = pd.concat(
    [pd.read_stata(f).assign(year_file=int(f.name[:4])) for f in files],
    ignore_index=True,
)
print(sig.shape)

In [ ]:
#Datastream
ri_sheets = pd.read_excel(DATA / "banks_data_bocconi" / "banks_ri.xlsm", sheet_name=None)
ri_sheets.pop("REQUEST_TABLE")

print(ri_sheets.keys())
print(ri_sheets["US"].shape)

In [ ]:
longs = []

for k, v in ri_sheets.items():
    long = v.melt(id_vars="Name", var_name="bank", value_name="ri")
    long = long.rename(columns={"Name": "date"})
    long["country"] = k
    longs.append(long)

ri = pd.concat(longs, ignore_index=True)

In [ ]:
print(ri.shape)
print(ri["country"].value_counts())
print(ri["bank"].nunique())

In [ ]:
print(f"{sig['uid_archive'].nunique():,} events, {len(sig):,} rows")
print(f"{sig['company'].nunique():,} companies in total") #nbr of unique company_name -> Number of companies
print(f"Average : {len(sig) / sig['uid_archive'].nunique():.1f} companies per event")

**Largest event:**

In [ ]:
top_uid = sig["uid_archive"].value_counts().index[0]
top = sig[sig["uid_archive"] == top_uid]
print(f"Largest event: {len(top)} firms named by {top['ngo_name1'].iloc[0]} on {top['date'].iloc[0]}")

Largest event. The event with the most targets is a single communication by ActionAid UK on 13 May 2013, naming 72 companies at once (BP, Shell, Tesco, AstraZeneca, …).

This matters for the event study: an NGO communication that names dozens of firms is unlikely to be firm-specific news, and any common price move that day would be absorbed by the market factor. We therefore compute n_targets (number of firms named in the same event) and will either drop or analyse separately the mass events above a threshold.

In [ ]:
counts = sig["uid_archive"].value_counts()          #number of firms per event
sig["n_targets"] = sig["uid_archive"].map(counts)   #attach it to every (event, firm) row

In [ ]:
events = sig.drop_duplicates("uid_archive")

print("n_targets, per (event, firm) row:")
print(sig["n_targets"].describe().round(2))

print("\nn_targets, per event:")
print(events["n_targets"].describe().round(2))

share_single = (events["n_targets"] == 1).mean()
print(f"\n{share_single:.1%} of events target a single company")

## A2. Clean Sigwatch and filter to US/UK/EU banks

In [ ]:
#Basic cleaning
sig["date"] = pd.to_datetime(sig["date"])                                    # dates are stored as text
for c in ["isin_1", "isin_2", "isin_3"]:
    sig[c] = sig[c].replace("", np.nan)                                      # empty strings -> missing
sig["parent_country"] = sig["company_parent_country"].replace({"USA": "US"}) # same country, two spellings

In [ ]:
# Names of the banks we have prices for (Datastream, incl. Swiss/Nordic file)

# Read the column headers of every country sheet and strip the " - TOT RETURN IND" suffix to get the bank name
def ds_names(file):
    sheets = pd.read_excel(DATA / "banks_data_bocconi" / file, sheet_name=None, nrows=1)
    sheets.pop("REQUEST_TABLE")
    return {c.rsplit(" - ", 1)[0].strip().lower() for v in sheets.values() for c in v.columns if c != "Name"}

# Combine the main file and the Swiss/Nordic file into one set of priced bank names
ds = ds_names("banks_ri.xlsm") | ds_names("scandinavian_banks_ri.xlsm")

In [ ]:
# Sigwatch name -> Datastream name, parsed from the Stata file
mp = {}
for line in open(DATA / "example_name_cleaning.txt", encoding="utf-8", errors="ignore"):
    if line.strip().startswith("replace company"):
        q = re.findall(r'"([^"]*)"', line)                      # [target, source1, source2, ...]
        mp.update({s.strip().lower(): q[0].strip().lower() for s in q[1:]})

# remove wrong mappings from the Stata file (e.g. Berkshire Hathaway -> a Swiss cantonal bank)
for k in ["berkshire hathaway", "pinnacle west capital corporation", "virgin group",
          "lloyd's of london", "ace group", "gruppo assicurazioni generali"]:
    mp.pop(k, None)

# Banco Popular Espanol has no price series: Datastream's "POPULAR" is Popular Inc. of Puerto Rico, a different bank
mp = {k: v for k, v in mp.items() if v != "popular"}

# fix targets that don't exist in Datastream + add banks the Stata file missed
mp.update({
    "bankinter, s.a.": "bankinter 'r'", "allianz se": "allianz", "allianz group / allianz se": "allianz",
    "jp morgan chase": "jp morgan chase & co.", "goldman sachs group, inc.": "goldman sachs gp.",
    "skandinaviska enskilda banken ab (seb)": "skandinaviska enskilda banken a",
    "handelsbanken": "svenska handelsbanken a", "dnb nor": "dnb", "u.s. bank": "us bancorp", "la caixa": "caixabank",
})

In [ ]:
# Match on the company, else on its parent (so subsidiaries like Halifax -> Lloyds are kept)
for c in ["company", "company_parent"]:
    s = sig[c].str.strip().str.lower()
    sig[c + "_ds"] = s.map(mp).fillna(s)
sig["ds_name"] = sig["company_ds"].where(sig["company_ds"].isin(ds),
                                         sig["company_parent_ds"].where(sig["company_parent_ds"].isin(ds)))

# Saga Group's Datastream index never changes (no price data), so its events cannot be studied
NO_PRICES = {"saga group"}
matched = sig[sig["ds_name"].notna() & ~sig["ds_name"].isin(NO_PRICES)].copy()

# Datastream's bank lists also contain insurers, an investment trust and a card network: these are not banks
NON_BANKS = {"allianz", "aegon", "asr nederland", "admiral group", "aflac", "alliance trust", "mastercard"}

# Project scope: US, UK and EU banks, so Swiss and Norwegian banks are out
EU = {"Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic", "Denmark", "Estonia", "Finland", "France",
      "Germany", "Greece", "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta", "Netherlands",
      "Poland", "Portugal", "Romania", "Slovakia", "Slovenia", "Spain", "Sweden"}
SCOPE = EU | {"US", "UK"}

banks = matched[~matched["ds_name"].isin(NON_BANKS) & matched["parent_country"].isin(SCOPE)].copy()

out = matched[~matched.index.isin(banks.index)]
print(f"{matched['ds_name'].nunique()} priced firms matched -> {banks['ds_name'].nunique()} US/UK/EU banks")
print(out.groupby(["parent_country", "ds_name"], observed=True).size().rename("rows dropped").to_string())

## A3. Name → ISIN lookup
One ISIN per bank. Datastream's ISIN comes first because it belongs to the series we price, then Sigwatch's. Three banks have neither and are filled by hand.

In [ ]:
# Datastream's ISIN per bank, from the list sheets of banks_info.xlsx
sheets = pd.read_excel(DATA / "banks_data_bocconi" / "banks_info.xlsx", sheet_name=None)
info = pd.concat([v for k, v in sheets.items() if k.startswith("LBANKS") and "ISIN CODE" in v], ignore_index=True)
info["ds_name"] = info["NAME"].astype(str).str.strip().str.lower()
ds_isin = info.dropna(subset=["ISIN CODE"]).drop_duplicates("ds_name").set_index("ds_name")["ISIN CODE"]

# Sigwatch's most frequent ISIN per bank
sig_isin = banks.dropna(subset=["isin_1"]).groupby("ds_name")["isin_1"].agg(lambda s: s.value_counts().index[0])

# No ISIN in either source
MANUAL_ISIN = {"citigroup": "US1729674242", "ally financial": "US02005N1000", "skandinaviska enskilda banken a": "SE0000148884"}

lookup = banks[["ds_name", "company", "company_parent", "parent_country"]].drop_duplicates().sort_values(["ds_name", "company"])
lookup["isin"] = lookup["ds_name"].map(ds_isin).fillna(lookup["ds_name"].map(sig_isin)).fillna(lookup["ds_name"].map(MANUAL_ISIN))
lookup["isin_sigwatch"] = lookup["ds_name"].map(sig_isin)                 # can differ: an ISIN changes after a merger or reverse split

one = lookup.drop_duplicates("ds_name")
print(f"{len(one)} banks | {one['isin'].isna().sum()} without an ISIN | "
      f"{(one['isin_sigwatch'].notna() & (one['isin'] != one['isin_sigwatch'])).sum()} where Sigwatch's ISIN differs from Datastream's")

#### **Description of the bank campaigns**

In [ ]:
# Descriptives of the bank campaigns

# Sigwatch gives one reach score per NGO (up to 5): keep the strongest one in the event
banks["ngo_power"] = banks[[f"ngo_power{i}" for i in range(1, 6)]].max(axis=1)

# Size of the sample: rows, events, banks and lead NGOs
print(f"{len(banks):,} rows | {banks['uid_archive'].nunique():,} events | "
      f"{banks['ds_name'].nunique()} banks | {banks['ngo_name1'].nunique():,} lead NGOs")

# Positive vs. negative campaigns
print(banks["sentiment"].map({-2: "very negative", -1: "negative", 0: "neutral", 1: "positive", 2: "very positive"})
      .value_counts().reindex(["very negative", "negative", "neutral", "positive", "very positive"]))

# Distribution of prominence, NGO reach and number of firms named in the same event
print(banks[["prominence", "ngo_power", "n_targets"]].describe().round(2))

# Most active NGOs
print(banks["ngo_name1"].value_counts().head(10))

**Sentiment**: The sample contains 3,212 campaign-bank observations (1,311 distinct events) targeting 59 listed banks, launched by 362 different lead NGOs. Campaigns are overwhelmingly critical: about 81% are negative or very negative (2,596 rows), versus about 14% positive (442) and 5% neutral (174). This is consistent with NGOs mostly using campaigns to pressure banks rather than to praise them, so the event study will have far more negative than positive events.

**Prominence**: Prominence is centred on 2, so a typical campaign mentions the bank in the body of the communication rather than the headline. The lead NGO’s reach is medium, with a quarter of the sample at 2.5 or above (global or international NGOs). n_targets is skewed: the median event names 8 firms but the maximum is 72. Many bank campaigns are therefore rankings or multi-company reports rather than bank-specific attacks. We flag events with more than 10 targets (mass) and treat them as a robustness check.

**Activity**: Activity is concentrated in a few NGOs: Friends of the Earth France alone accounts for 240 rows (about 7% of the sample), followed by Which? (UK, 177) and Rainforest Action Network (US, 120). The top ten are mostly national or regional groups (France, UK, US, Belgium, Netherlands, Spain), so the results may reflect these NGOs’ repeated campaigns rather than the whole NGO universe.

#### **One row per bank-day, then flag problem events**

**One row per bank-day**
A bank can appear several times on the same day (parent and subsidiary, or several NGOs), so we collapse to one row per bank and date. We average sentiment, keep the maximum prominence and NGO reach, and add a `climate` flag from the issue fields.

In [ ]:
#One row per bank-day
issues = banks[["issue_name1", "issue_name2", "issue_name3"]].fillna("").agg(" ".join, axis=1)
banks["climate"] = issues.str.contains(r"climate|carbon|coal|fossil|(?<!palm )\boil|\bgas\b|pipeline|fracking|emission|renewable", case=False)

events = (banks.sort_values(["ds_name", "date"])
          .groupby(["ds_name", "date"], as_index=False)
          .agg(uid_archive=("uid_archive", "first"), uids=("uid_archive", lambda s: ";".join(map(str, s.unique()))),
               parent_country=("parent_country", "first"), ngo=("ngo_name1", "first"), n_rows=("uid_archive", "size"),
               sentiment=("sentiment", "mean"), min_sentiment=("sentiment", "min"),
               prominence=("prominence", "max"), ngo_power=("ngo_power", "max"),
               n_targets=("n_targets", "min"), climate=("climate", "max")))
events["negative"] = (events["sentiment"] < 0).astype(int)
events["isin"] = events["ds_name"].map(one.set_index("ds_name")["isin"])

**Flag problem events and select the final sample**
For the event study we need clean events. We (1) drop days where the same bank got opposite-sign news within 5 days, since we can't tell which news moved the price, and (2) keep only the first event in each 30-day run, so the [-10, +10] windows of kept events don't overlap. Events naming more than 10 firms (`mass`) are only flagged, not dropped.

The overlap rule stays here for now because Part C reads `events["keep"]`. `gap` is an argument, so other rules can be tested.

In [ ]:
# Flag problem events and keep a clean, non-overlapping sample
g = events.groupby("ds_name")
events["days_since_prev"] = g["date"].diff().dt.days

# Bank-days with opposite-sign news within 5 days (can't tell which moved the price)
events["opposite"] = (events["days_since_prev"] <= 5) & (events["sentiment"] * g["sentiment"].shift() < 0)
events["opposite"] |= events["opposite"].groupby(events["ds_name"]).shift(-1, fill_value=False)
events["mass"] = events["n_targets"] > 10                    # flag only, not dropped

# Keep an event only if it is >= gap days after the last kept one
def first_in_window(dates, gap=30):
    keep, last = [], None
    for d in dates:
        ok = last is None or (d - last).days >= gap
        keep.append(ok)
        last = d if ok else last
    return keep

# Apply it bank by bank, after removing the opposite-sign events
def overlap_rule(events, gap=30):
    ok = events[~events["opposite"]]
    keep = pd.Series(False, index=events.index)
    keep.loc[ok.index] = ok.groupby("ds_name")["date"].transform(lambda s: pd.Series(first_in_window(s, gap), index=s.index))
    return keep.astype(bool)


events["keep"] = overlap_rule(events)
print(events[["opposite", "mass", "keep", "climate"]].sum(), len(events))

Out of 2,935 bank-days, 231 (8%) have opposite-sign news within 5 days, 1,013 (35%) are flagged as mass events and 1,135 (39%) carry a climate tag. After removing the opposite-sign days, the 30-day rule leaves 1,363 events with non-overlapping [−10, +10] windows. We keep `mass` and `climate` as flags, so later we can test whether the effect differs for bank-specific vs. multi-firm campaigns, and for climate vs. other campaigns, before and after COP21.

In [ ]:
# Map each event to a trading day
def calendar(file):                                                      # trading days from the Fama-French file
    d = pd.read_excel(DATA / "banks_data_bocconi" / file, usecols=["date"])["date"]
    return pd.to_datetime(d, format="%m/%d/%Y").sort_values().to_frame("trade_date")

cal = {"US": calendar("North_America_3_Factors_Daily.xlsx"), "EU": calendar("Europe_3_Factors_Daily.xlsx")}
events["region"] = np.where(events["parent_country"] == "US", "US", "EU")     # US banks: US calendar, others: Europe

# event day = first trading day on or after the campaign date
events = pd.concat([pd.merge_asof(g.sort_values("date"), cal[r], left_on="date", right_on="trade_date", direction="forward")
                    for r, g in events.groupby("region")]).sort_values(["ds_name", "date"]).reset_index(drop=True)
events["shifted"] = events["trade_date"] != events["date"]
print(events["shifted"].sum(), "of", len(events), "events moved to the next trading day")

**Comment:** All 2,935 bank-days already fall on trading days (Monday to Friday, none on a holiday), so no event needed to be moved. The column `trade_date` is kept as the event day for the event study.

### **Figure 1: When, against whom, by whom**
We plot all bank-days with a campaign (before the 30-day rule) to see how the campaigns are spread over time, across banks and across NGOs.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import matplotlib.dates as mdates

# Palette: red = negative, green = positive, grey = neutral (same as the PC Labs)
C = {"neg": "#c8283c", "neu": "#b7b5ad", "pos": "#1f8a5b", "navy": "#0d3b78", "grey": "#52514e"}
events["sign"] = np.select([events["sentiment"] < 0, events["sentiment"] > 0], ["neg", "pos"], "neu")
COP21 = pd.Timestamp("2015-12-12")                         # Paris Agreement

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), gridspec_kw={"width_ratios": [1.3, 1, 1.1], "wspace": 0.95})

# A. campaigns per quarter by tone
q = events.groupby([events["date"].dt.to_period("Q").dt.to_timestamp(), "sign"]).size().unstack(fill_value=0)
ax = axes[0]; bottom = np.zeros(len(q))
for s, lab in [("neg", "Negative"), ("neu", "Neutral"), ("pos", "Positive")]:
    ax.bar(q.index, q[s], width=70, bottom=bottom, color=C[s], label=lab); bottom += q[s].values
ax.axvline(COP21, color="black", ls=":", lw=1); ax.text(COP21, ax.get_ylim()[1] * 0.97, " COP21", va="top", fontsize=9)
ax.set_title("A. Campaigns per quarter, by tone", loc="left", fontweight="bold")
ax.set_ylabel("Bank-days with a campaign"); ax.legend(frameon=False, loc="upper left", fontsize=9)

# B. most targeted banks
top = events.groupby("ds_name").size().nlargest(12).index
b = events[events["ds_name"].isin(top)].groupby(["ds_name", "sign"]).size().unstack(fill_value=0).reindex(top)[::-1]
ax = axes[1]; left = np.zeros(len(b))
for s in ["neg", "neu", "pos"]:
    ax.barh(b.index.str.title(), b[s], left=left, color=C[s], height=0.7); left += b[s].values
ax.set_title("B. 12 most targeted banks", loc="left", fontweight="bold"); ax.set_xlabel("Bank-days with a campaign")

# C. most active NGOs
ngo_top = events.groupby("ngo").size().nlargest(10)[::-1]
ax = axes[2]; ax.barh([x[:26] + ("…" if len(x) > 26 else "") for x in ngo_top.index], ngo_top.values, color=C["navy"], height=0.7)
ax.set_title("C. 10 most active lead NGOs", loc="left", fontweight="bold"); ax.set_xlabel("Bank-days with a campaign")

for ax in axes: ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25, axis="x" if ax is not axes[0] else "y"); ax.set_axisbelow(True)

# Panel A: one x label every 2 years (must come after the figure is created)
axes[0].xaxis.set_major_locator(mdates.YearLocator(2))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

share = (events["sign"] == "neg").mean()
fig.suptitle(f"Figure 1: Bank campaigns are mostly negative ({share:.0%}) and concentrated in a few banks and NGOs",
             x=0.01, ha="left", fontsize=13.5, fontweight="bold")
fig.text(0.01, -0.02, f"{len(events):,} bank-days, {events['ds_name'].nunique()} banks, 2010-2018. "
         "One row per bank and day (all sentiments, before the 30-day rule). COP21 = Paris Agreement, 12 Dec 2015.", fontsize=9, color=C["grey"])
plt.show()

**Comment:**

**A.** Campaigns against banks rise from about 20-80 per quarter in 2010–2011 to peaks of 120–155 in 2017–2018, and red (negative) dominates in every quarter. The positive share is largest in 2015 (about 23%), the year of COP21, when NGOs also praised banks for climate commitments.

**B.** A few large banks are the main targets: HSBC (214 bank-days), BNP Paribas (201) and Deutsche Bank (193). Deutsche Bank is almost purely negative, while BNP Paribas and ING also receive a visible share of positive campaigns.

**C.** Activity is concentrated in a handful of NGOs: Friends of the Earth France leads with 224 bank-days, followed by Which? (UK) at 131 and Rainforest Action Network at 109. Most are national or regional groups (France, UK, US, Belgium, Netherlands, Spain), so a few NGOs' repeated campaigns drive much of the sample.

### Figure 2: What a campaign looks like
We describe the three Sigwatch intensity variables: where the bank is mentioned (`prominence`), the reach of the strongest NGO (`ngo_power`) and how many firms are named in the same communication (`n_targets`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw={"wspace": 0.3})

# A. prominence, negative vs positive campaigns (share within each tone)
ax = axes[0]; w = 0.38; lv = [1, 2, 3, 4]
for k, (s, lab) in enumerate([("neg", "Negative"), ("pos", "Positive")]):
    sh = events[events["sign"] == s]["prominence"].value_counts(normalize=True).reindex(lv, fill_value=0)
    ax.bar(np.array(lv) + (k - 0.5) * w, sh.values, width=w, color=C[s], label=lab)
ax.set_xticks(lv, ["1\nreport only", "2\nelsewhere", "3\nfirst paragraph", "4\nheadline"], fontsize=9)
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax.legend(frameon=False)
ax.set_title("A. Where is the bank mentioned?", loc="left", fontweight="bold"); ax.set_ylabel("Share of campaigns")

# B. reach of the strongest NGO
ax = axes[1]; p = events["ngo_power"].value_counts(normalize=True).sort_index()
ax.bar(p.index.astype(str), p.values, color=C["navy"], width=0.7)
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax.set_xlabel("ngo_power (0.5 = local, 2.75 = global)")
ax.set_title("B. Reach of the strongest NGO", loc="left", fontweight="bold")

# C. firms named in the same communication
ax = axes[2]; clip = events["n_targets"].clip(upper=30)
ax.hist(clip, bins=np.arange(0.5, 31.5, 1), color=C["navy"], weights=np.ones(len(clip)) / len(clip))
ax.axvline(10.5, color=C["neg"], ls="--", lw=1.2)
ax.text(11, ax.get_ylim()[1] * 0.92, f"{events['mass'].mean():.0%} of campaigns\nname more than 10 firms\n(flagged 'mass')", color=C["neg"], fontsize=9, va="top")
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax.set_xlabel("Firms named in the same communication (30 = 30 or more)")
ax.set_title("C. Many campaigns are not bank-specific", loc="left", fontweight="bold")

for ax in axes: ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25, axis="y"); ax.set_axisbelow(True)
fig.suptitle("Figure 2: A typical campaign is mid-prominence, from a mid-reach NGO, and often names many firms",
             x=0.01, ha="left", fontsize=13.5, fontweight="bold", y=1.03)
plt.show()

**Comment:**

**A.** Negative campaigns put the bank in more visible places than positive ones: about 24% mention it in the headline versus 15% of positive campaigns. Positive news is mostly mentioned elsewhere in the text (about 61%).

**B.** The reach of the lead NGO is bimodal: about 36% of campaigns come from NGOs with reach 1.0 (local or national groups) and 32% from NGOs at 2.5 (international groups). Very local (0.5) and very global (2.75) NGOs are rare.

**C.** Most campaigns name few firms (a peak at 5), but 35% name more than 10, and a small group names 30 or more. These are bank rankings and multi-company reports, not bank-specific news, so we flag them as `mass` and use them as a robustness check.

## A4. Handoff files
- `events_banks.csv`: one row per bank-day, with the flags and the ISIN. **C uses it for the event study.** `uids` links each row back to its campaigns.
- `campaigns_banks.csv`: one row per campaign, with the report text and links. **D uses it for the add-on, and A for the NLP step.**
- `lookup_banks.csv`: Sigwatch name → Datastream name → ISIN.

In [ ]:
# One row per campaign, with the text and links
campaigns = (banks.groupby("uid_archive", as_index=False)
             .agg(date=("date", "first"), ngo=("ngo_name1", "first"), ngo_power=("ngo_power", "max"),
                  sentiment=("sentiment", "mean"), prominence=("prominence", "max"), climate=("climate", "max"),
                  n_targets=("n_targets", "first"), n_banks=("ds_name", "nunique"),
                  banks=("ds_name", lambda s: "; ".join(sorted(set(s)))),
                  issue_name1=("issue_name1", "first"), issue_name2=("issue_name2", "first"), issue_name3=("issue_name3", "first"),
                  report=("report", "first"), link1=("link1", "first"), link2=("link2", "first"), link3=("link3", "first"),
                  link4=("link4", "first"), link5=("link5", "first")))
campaigns["report"] = campaigns["report"].fillna("").str.replace(r"<[^>]+>", " ", regex=True).map(html.unescape).str.strip()   # HTML -> plain text

events.to_csv(OUT / "events_banks.csv", index=False)
campaigns.to_csv(OUT / "campaigns_banks.csv", index=False)
lookup.to_csv(OUT / "lookup_banks.csv", index=False)
print(f"{len(events):,} bank-days | {len(campaigns):,} campaigns | {(campaigns['report'].str.len() > 0).mean():.0%} with report text, "
      f"{(campaigns['link1'].fillna('').str.len() > 0).mean():.0%} with a link")

---
# Part B: Market data and betas (owner: B, Alissa)
Build the daily panel (returns from RI, Fama-French factors, risk-free rate, currencies), then estimate rolling CAPM and Fama-French betas (task #4), which give each bank's expected return on each day.

**Handoff to C:** `handoff/panel_returns.csv`, `handoff/betas_daily.csv`.

## B1. Daily panel
We build one daily panel with returns for the 59 banks in our events, plus the Fama-French factors and the risk-free rate. US banks use the North America factors and all other banks the Europe factors. We keep only banks that appear in the campaign data.

In [ ]:
# Total-return indices of all banks (main file + Swiss/Nordic file), same bank key as the events file
def load_ri(file):
    sheets = pd.read_excel(DATA / "banks_data_bocconi" / file, sheet_name=None)
    sheets.pop("REQUEST_TABLE")
    return pd.concat([v.rename(columns={"Name": "date"}).melt(id_vars="date", var_name="bank", value_name="ri").assign(country=k)
                      for k, v in sheets.items()], ignore_index=True)

ri = pd.concat([load_ri("banks_ri.xlsm"), load_ri("scandinavian_banks_ri.xlsm")], ignore_index=True)
ri["date"] = pd.to_datetime(ri["date"], errors="coerce")
ri["ds_name"] = ri["bank"].str.rsplit(" - ", n=1).str[0].str.strip().str.lower()
ri = ri[ri["ds_name"].isin(events["ds_name"])].dropna(subset=["date", "ri"]).sort_values(["ds_name", "date"]).drop_duplicates(["ds_name", "date"])

In [ ]:
# Daily returns; drop impossible moves (> +/-100% in a day = data errors) and flag very large ones (> 50%)
ri["ret"] = ri.groupby("ds_name")["ri"].pct_change()
ri = ri.dropna(subset=["ret"])
ri["ret"] = ri["ret"].where(ri["ret"].abs() <= 1)
ri["big"] = ri["ret"].abs() > 0.5

**Risk-free rate:** the `RF` column of the factor files is rounded to 0.01% a day, so it is exactly 0 on more than three quarters of days since 2008 and overstates the average rate (0.82% a year, against 0.55% for the T-bill). We use the daily 4-week T-bill from `risk_free_rate.xlsm`, the closest match to the 1-month bill behind the Fama-French factors.

In [ ]:
# Daily 4-week US T-bill, annualised % -> daily decimal (actual/360)
rf = pd.read_excel(DATA / "banks_data_bocconi" / "risk_free_rate.xlsm", sheet_name="DAILY", header=2)
rf.columns = ["date", "tb3m", "tb4w"]
rf["date"] = pd.to_datetime(rf["date"], errors="coerce")
rf = rf.dropna(subset=["date"]).assign(rf=lambda d: pd.to_numeric(d["tb4w"]) / 100 / 360)[["date", "rf"]]

**Factors:** the data pack only has 3-factor files, so `ff` below means FF3 for now. To move to FF5, put `North_America_5_Factors_Daily.xlsx` and `Europe_5_Factors_Daily.xlsx` (same format as the 3-factor files) in `banks_data_bocconi/`: the panel, the betas and C's `model="ff"` then use all five factors with no code change.

In [ ]:
# Fama-French factors: % -> decimals, US banks use North America, all others Europe. The 5-factor file is used once it is in the folder
def load_ff(name, region):
    files = [DATA / "banks_data_bocconi" / f"{name}_{n}_Factors_Daily.xlsx" for n in (5, 3)]
    ff = pd.read_excel(next(f for f in files if f.exists()))
    ff["date"] = pd.to_datetime(ff["date"], format="%m/%d/%Y")
    ff.columns = ff.columns.str.lower().str.replace("-", "_")
    factors = ff.columns.drop(["date", "rf"])
    ff[factors] /= 100
    return ff.drop(columns="rf").assign(region=region)

ff = pd.concat([load_ff("North_America", "US"), load_ff("Europe", "EU")], ignore_index=True)
FACTORS = list(ff.columns.drop(["date", "region"]))
ri["region"] = np.where(ri["country"] == "US", "US", "EU")
print("Factors:", FACTORS)

**Currencies:** the Fama-French factors are in USD, but the bank series are in local currency: the Datastream request is plain `RI`, with no currency conversion. The `RF` column is also identical in the US and Europe factor files, so the European factors are USD returns over a US rate. The data pack has no European risk-free rate or factors in euro, so we convert bank returns to USD, which makes the US T-bill the right risk-free rate for every bank. This is the slides' warning that the risk-free rate differs across markets.

**Exchange rates:** `banks_data_bocconi/fx_usd.csv` is built from the ECB's daily euro reference rates (series `EXR.D.USD.EUR.SP00.A`, `EXR.D.GBP.EUR.SP00.A`, `EXR.D.SEK.EUR.SP00.A`), not part of the data pack. GBP and SEK are crossed through the euro. On days with no ECB rate, the last rate is carried forward.

**Check:** on 24 June 2016 (Brexit), HSBC's return goes from −1.4% in sterling to −9.2% in USD, in line with the USD Europe factor at −8.8%.

**Limitation:** the ECB fixes its rates at 14:15 CET, before European markets close, so the FX move and the stock move of a day are not perfectly aligned.

In [ ]:
# Align currencies: bank returns -> USD, using fx_usd.csv (columns: date, EUR, GBP, SEK; USD per 1 unit of local currency)
CURRENCY = {"US": "USD", "UK": "GBP", "SD": "SEK"}                        # every other sheet in the sample is euro area
ri["currency"] = ri["country"].map(CURRENCY).fillna("EUR")
ri["ret_local"] = ri["ret"]                                               # kept for the stale-price check: FX moves would hide a zero return

fx_file = DATA / "banks_data_bocconi" / "fx_usd.csv"
USD_ALIGNED = fx_file.exists()
if USD_ALIGNED:
    fx = pd.read_csv(fx_file, parse_dates=["date"]).set_index("date").sort_index()
    fx = fx.reindex(pd.bdate_range(fx.index.min(), fx.index.max())).ffill().pct_change()       # FX holidays: rate carried forward
    fx = fx.rename_axis("date").reset_index().melt(id_vars="date", var_name="currency", value_name="fx_ret")
    ri = ri.merge(fx, on=["date", "currency"], how="left")
    ri["ret"] = (1 + ri["ret"]) * (1 + ri["fx_ret"].where(ri["currency"] != "USD", 0)) - 1
    print("Bank returns converted to USD")
else:
    print(f"WARNING: no {fx_file.name}. {ri.loc[ri['currency'] != 'USD', 'ds_name'].nunique()} non-US banks stay in local currency against USD factors")

In [ ]:
# Final panel: one row per bank and day, with excess return (return - risk-free) and a trading-day counter t
panel = ri.merge(ff, on=["date", "region"], how="inner").merge(rf, on="date", how="left")
assert panel["rf"].notna().all()
panel["exret"] = panel["ret"] - panel["rf"]
panel["mkt"] = panel["mkt_rf"] + panel["rf"]                              # market return
panel = panel[["date", "ds_name", "country", "currency", "region", "ri", "ret_local", "ret", "exret", *FACTORS, "mkt", "rf", "big"]].dropna(subset=["ret"])
panel = panel.sort_values(["ds_name", "date"]).reset_index(drop=True)
panel["t"] = panel.groupby("ds_name").cumcount()

# The region must agree between the events (from the parent's country) and the panel (from the Datastream sheet)
reg = events.drop_duplicates("ds_name").set_index("ds_name")["region"]
assert (reg == panel.groupby("ds_name")["region"].first().reindex(reg.index)).all()

panel.to_csv(OUT / "panel_returns.csv", index=False)
print(f"{len(panel):,} bank-days | {panel['ds_name'].nunique()} banks | {panel['date'].min().date()} to {panel['date'].max().date()}")
print(panel.groupby("region")["ds_name"].nunique())
print(panel.drop_duplicates("ds_name")["currency"].value_counts())

**Comment:** The panel has 172,394 daily observations for 59 banks (21 US, 38 European: 28 in euro, 7 in sterling, 3 in Swedish krona) from January 2008 to September 2020. The panel holds the total-return index `ri`, the bank return `ret`, the market return `mkt` and the risk-free rate `rf`, plus the factors. `ret` is in USD, while `ri` and `ret_local` stay in the original currency. Every bank in the campaign data has market data, so no bank is lost at this stage. Some events fall outside a bank's price history or lack a full estimation window, and C handles them in C2.

### Figures 3-4: How the bank panel behaves
We build an equal-weight portfolio of the banks in each region and compare it with the market. Figure 3 shows performance and Figure 4 shows how risky the banks are (beta) and how reliable their prices are.

In [ ]:
# Equal-weight bank portfolio, market, bank/market ratio and rolling 250-day beta, per region
plot = panel[panel["date"].between("2008-01-03", "2020-04-28")]           # the main price file stops on 28 Apr 2020
REG = {"US": ("US banks", C["navy"]), "EU": ("European banks", C["neg"])}
EVENTS = {"Lehman": "2008-09-15", "Euro crisis": "2011-08-01", "COP21": "2015-12-12", "Covid": "2020-03-11"}
res = {}
for r in REG:
    d = plot[plot["region"] == r]
    bank = d.groupby("date")["ret"].mean()
    mkt = d.groupby("date")[["mkt_rf", "rf"]].first().sum(axis=1)                    # market return = Mkt-RF + RF
    res[r] = pd.DataFrame({"bank": 100 * (1 + bank).cumprod(), "mkt": 100 * (1 + mkt).cumprod(),
                           "beta": bank.rolling(250).cov(mkt) / mkt.rolling(250).var()})
    res[r]["rel"] = 100 * res[r]["bank"] / res[r]["mkt"]

import matplotlib.ticker
def marks(ax):                                                            # vertical lines for the key episodes
    for lab, d in EVENTS.items():
        left = lab in ("Lehman", "Euro crisis")
        ax.axvline(pd.Timestamp(d), color="black", ls=":", lw=0.9)
        ax.text(pd.Timestamp(d), 1.0, (" " + lab) if left else (lab + " "), transform=ax.get_xaxis_transform(),
                fontsize=8.5, va="top", ha="left" if left else "right")
def style(axes):
    for ax in axes: ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25); ax.set_axisbelow(True)

In [ ]:
# Figure 3 (performance)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"wspace": 0.3})

# A. levels, with end values labelled
ax = axes[0]
for r, (lab, col) in REG.items():
    ax.plot(res[r].index, res[r]["bank"], color=col, lw=1.6)
    ax.plot(res[r].index, res[r]["mkt"], color=col, lw=1.1, ls="--", alpha=0.55)
    ax.annotate(f"{lab}\n{res[r]['bank'].iloc[-1]:.0f}", (res[r].index[-1], res[r]["bank"].iloc[-1]), xytext=(5, 0), textcoords="offset points", fontsize=8.5, color=col, va="center")
    ax.annotate(f"market {res[r]['mkt'].iloc[-1]:.0f}", (res[r].index[-1], res[r]["mkt"].iloc[-1]), xytext=(5, 0), textcoords="offset points", fontsize=8, color=C["grey"], va="center")
ax.set_yscale("log"); ax.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set_yticks([30, 50, 100, 200]); ax.set_yticklabels(["30", "50", "100", "200"]); ax.set_xlim(right=pd.Timestamp("2022-01-01"))
marks(ax); ax.set_title("A. Banks (solid) vs market (dashed), 100 in Jan 2008", loc="left", fontweight="bold")

# B. bank / market ratio
ax = axes[1]
for r, (lab, col) in REG.items(): ax.plot(res[r].index, res[r]["rel"], color=col, lw=1.6, label=lab)
ax.axhline(100, color=C["grey"], lw=0.8); marks(ax)
ax.set_title("B. Banks relative to their market (100 = equal)", loc="left", fontweight="bold"); ax.legend(frameon=False, loc="lower center")

style(axes)
fig.suptitle("Figure 3: Banks fell harder than the market in crises, and European banks lag their market since 2010",
             x=0.01, ha="left", fontsize=13.5, fontweight="bold")
n_end = (panel.groupby("ds_name")["date"].max() <= "2020-04-28").sum()
cov = plot.groupby(["date", "region"])["ds_name"].nunique().unstack()
fig.text(0.01, -0.02, f"Equal-weight portfolio of the {panel['ds_name'].nunique()} event banks; market = Mkt-RF + RF. Plot stops on 28 Apr 2020, when {n_end} of {panel['ds_name'].nunique()} banks end. "
         f"Banks with data: {cov['EU'].min()}-{cov['EU'].max()} European, {cov['US'].min()}-{cov['US'].max()} US.", fontsize=8.5, color=C["grey"])
plt.show()

**Comment:** All lines are in USD.

**A.** Bank stocks fall much further than the market in every crisis. In 2008-09 the US bank portfolio loses about 70% and the European one about 80%, and both fall sharply again in the euro crisis (2011-12) and in the Covid crash (March 2020). Between the crises the two regions diverge. US banks stay above their market for most of 2017-2019, but after the Covid crash they stand at 176 against 234 for the market on 28 April 2020. European banks never get back to 100, staying around or below 80 after 2010, while their market rises to about 130 before Covid, and they end at 30 against 105. The COP21 line falls in a flat period for both, with no visible reaction at aggregate level.

**B.** The ratio makes the gap clear. US banks move between about 80 and 110 from 2011 until 2016, jump to about 120 after the US election at the end of 2016 (which matters for the Trump effect we test later), peak near 125 in 2018 and fall to about 75 in April 2020. European banks peak near 115 in 2009, fall to about 50 in 2012, recover to about 70 in 2014, and then drift down to about 38 before Covid and 27 in April 2020. The underperformance of European banks is persistent, not a one-off shock.

In [ ]:
#Figure 4 (risk and data quality)
# Share of days with exactly zero return per bank (stale prices), from 2010
zero = (panel[panel["date"] >= "2010-01-01"].assign(z=lambda d: d["ret_local"].abs() < 1e-12)
        .groupby("ds_name")["z"].mean().sort_values(ascending=False))
reg_of = panel.groupby("ds_name")["region"].first()
top = zero.head(12)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"wspace": 0.55, "width_ratios": [1.2, 1]})

# A. rolling beta of the bank portfolio
ax = axes[0]
for r, (lab, col) in REG.items(): ax.plot(res[r].index, res[r]["beta"], color=col, lw=1.6, label=lab)
ax.axhline(1, color=C["grey"], lw=0.8); marks(ax)
ax.set_title("A. Bank portfolio beta (rolling 250 days)", loc="left", fontweight="bold")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.55, 0.92))

# B. banks with the most stale prices
ax = axes[1]
ax.barh([s[:26] for s in top.index], top.values * 100, color=[REG[reg_of[s]][1] for s in top.index], height=0.65)
ax.set_title("B. Banks with the most zero-return days (2010-20)", loc="left", fontweight="bold")
ax.set_xlabel("% of trading days with zero return")

style(axes)
fig.suptitle("Figure 4: Bank betas move a lot over time, and a few banks have stale prices", x=0.01, ha="left", fontsize=13.5, fontweight="bold")
fig.text(0.01, -0.02, "Beta of the equal-weight bank portfolio on the market. Bars: red = European, blue = US bank. A zero return means the price did not change that day.",
         fontsize=8.5, color=C["grey"])
plt.show()

**Comment:**

**A.** The beta of the bank portfolio is far from constant. For US banks it ranges from about 0.9 to about 2.5: it peaks at about 2.5 at the start of 2010 after the financial crisis, falls to about 1.3 in 2011, rises again to about 1.6 in 2017, and falls back to about 0.9 in 2019 before rising with Covid. European banks mostly move between 1.0 and 1.6, peak at about 1.7 at the end of 2009 and reach a minimum of about 0.76 in 2018. They are close to US banks for much of the period, and above them in 2013-2015. A beta estimated in one period is therefore a poor guide to another, so each event must get its own beta from a window just before it (the 250 days ending 50 days before the event), not one beta for the whole sample.

**B.** A few banks have prices that do not change on many days (measured on local-currency returns, because FX moves would hide a zero return). Monte dei Paschi has a zero return on about 34% of days and Permanent TSB on about 16%. Most banks are far lower (the median is about 4.5%). Ten of the twelve banks shown are European (the exceptions are Ally Financial and Synovus). For these banks the beta is estimated with more noise, and an event day with no price change can hide a real reaction. We should check that these banks do not drive the results. We also dropped Saga Group, whose Datastream index never changes (100% zero returns), because it has no usable price data, and removed a duplicated Crédit Agricole series that created false zero returns.

## B2. Rolling betas and expected returns (#4)
For each bank and each day `t`, we regress the bank's excess return on the factors over the window [t−300, t−50], which ends 50 days before `t` as in Krüger (2015), so the estimate never uses the day it is applied to.

`betas_daily.csv` has one row per bank-day, with the parameters (`capm_alpha`, `capm_mkt_rf`, `ff_alpha`, `ff_mkt_rf`, `ff_smb`, `ff_hml`) and the expected excess return for that day (`exp_capm`, `exp_ff`). The abnormal return of a day is `exret - exp_capm`.

**For C:** in the event study, read the parameters at the event day `t0` and hold them fixed over the whole event window, as described in the Part C notes.

In [ ]:
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS

EST_START, EST_END = 300, 50                                              # estimation window [t-300, t-50]

def rolling_params(d, factors, name):
    window = EST_START - EST_END + 1
    fit = RollingOLS(d["exret"], sm.add_constant(d[factors]), window=window).fit(params_only=True)
    p = fit.params.shift(EST_END)                                         # window ends EST_END days before t
    expected = p["const"] + (p[factors] * d[factors]).sum(axis=1, min_count=len(factors))
    return p.rename(columns=lambda c: f"{name}_alpha" if c == "const" else f"{name}_{c}").assign(**{f"exp_{name}": expected})

parts = []
for _, d in panel.groupby("ds_name"):
    parts.append(pd.concat([d[["date", "ds_name", "region", "t", "exret"]],
                            rolling_params(d, ["mkt_rf"], "capm"), rolling_params(d, FACTORS, "ff")], axis=1))
betas = pd.concat(parts, ignore_index=True).dropna(subset=["capm_alpha"])

betas.to_csv(OUT / "betas_daily.csv", index=False)
print(f"{len(betas):,} bank-days with betas | {betas['ds_name'].nunique()} banks")
print(betas.groupby("region")[["capm_mkt_rf", "ff_mkt_rf", "ff_smb", "ff_hml"]].median().round(2))
print(f"Abnormal return, daily std: CAPM {(betas['exret'] - betas['exp_capm']).std():.4f}, FF {(betas['exret'] - betas['exp_ff']).std():.4f}")

**Comment:** We get betas for 154,694 bank-days and all 59 banks (a bank needs 300 trading days of history first). The median CAPM beta is 1.26 for US banks and 1.31 for European banks. In the Fama-French model both load strongly on HML (1.00 and 1.56), so banks behave like value stocks, which is why the market beta of European banks drops to 0.93 once HML and SMB are included. The extra factors only reduce the daily abnormal-return volatility from 2.32% to 2.21%, so CAPM and Fama-French CARs should be close. The betas match separate OLS regressions on a random sample of 1,500 bank-days.

**Effect of the USD conversion:** for the 921 European events, the median beta rises from 1.24 to 1.38 and the median R² from 0.39 to 0.42, so the factors fit better once returns and factors share a currency. CARs change little overall (correlation 0.97 with the local-currency CARs, largest change 6.8 points). The 380 US events are unchanged.

## 3. Event study: CAPM and cumulative abnormal returns

We follow Krueger (2015). For each kept campaign event, we estimate a market model for the targeted bank just before the event, use it to predict the bank's "normal" return around the event, and measure how far the actual return deviates from it.

In [ ]:
ev = events[events["keep"]].copy()

ev = ev.merge(
    panel[["ds_name", "date", "t"]], 
    left_on=["ds_name", "trade_date"],
    right_on=["ds_name", "date"],
    how="left",  
    suffixes=("", "_panel"),
)
ev = ev.rename(columns={"t": "t0"})

In [ ]:
print(f"{len(ev):,} events kept")
print(f"{ev['t0'].isna().sum()} without a matching trading day in the panel")
print(f"{(ev['t0'] < 300).sum()} with less than 300 days of history before t0")

In [ ]:
miss = ev[ev["t0"].isna()]
span = panel.groupby("ds_name")["date"].agg(first="min", last="max")
miss = miss.join(span, on="ds_name")

outside = (miss["trade_date"] < miss["first"]) | (miss["trade_date"] > miss["last"])
print(f"{outside.sum()} outside the bank's price history (not yet listed / delisted)")
print(f"{(~outside).sum()} inside the history but no return that day (holiday / removed return)")
print(f"{miss['ds_name'].nunique()} distinct banks concerned")

**Locating the event day:** We number each bank’s trading days (t) so that all windows are measured in trading days, and find the event day t0 by matching each kept event to the panel on bank and trade date. Of the 1,478 kept events, 57 (8 banks) fall outside the bank’s price history: the bank was not yet listed, or no longer listed, on the event date, so no market reaction can be measured. No event is lost to holidays or removed returns. A further 14 events have fewer than 300 trading days before t0, so the estimation window [t0−300, t0−50] cannot be filled. Dropping both leaves 1,407 events (95% of the kept sample).

In [ ]:
ev = ev[ev["t0"].notna() & (ev["t0"] >= 300)].copy()
ev["t0"] = ev["t0"].astype(int)          #t0 was float because of the NaNs
print(f"{len(ev):,} events for the event study")

**Method:** All windows are measured in the bank's trading days (`t`), not calendar days.

- **Estimation window** [t0−300, t0−50] (251 days). We regress the bank's excess return on the market excess return (North America factors for US banks, Europe factors for the others):
  $r_{i,t} - r_{f,t} = \alpha_i + \beta_i \,(r_{m,t} - r_{f,t}) + \varepsilon_{i,t}$.
  The 50-day gap keeps any pre-event information leakage out of the estimation.
- **Abnormal return** on each day of the event window [t0−10, t0+10]:
  $AR_{i,t} = (r_{i,t} - r_{f,t}) - (\hat\alpha_i + \hat\beta_i \,(r_{m,t} - r_{f,t}))$.
- **CAR** = sum of the 21 daily ARs. We also keep the event-day abnormal return `ar0`.

Because `t0 ≥ 300` and `t` has no gaps, every estimation window has exactly 251 days. The loop also checks that the event window is complete (21 days), which matters only if a bank's price series ends before t0+10; this never happens here, so all 1,407 events get a CAR.

In [ ]:
# all campaign days of each bank, in trading days, including those dropped by the overlap rule
camp = events.merge(panel[["ds_name", "date", "t"]], left_on=["ds_name", "trade_date"],
                    right_on=["ds_name", "date"], how="inner", suffixes=("", "_panel"))
campaign_t = camp.groupby("ds_name")["t"].apply(lambda s: np.sort(s.unique()))

In [ ]:
MODELS = {"capm": ["mkt_rf"], "ff": FACTORS}


def run_event_study(ev, panel, betas, pre=10, post=10, model="capm"):

    factors = MODELS[model]
    cols = [f"{model}_alpha"] + [f"{model}_{f}" for f in factors]
    data = ev.merge(betas[["ds_name", "t", *cols]], left_on=["ds_name", "t0"], right_on=["ds_name", "t"], how="left")

    rows = []
    for _, e in data.iterrows():
        bank = panel[panel["ds_name"] == e["ds_name"]]
        win = bank[(bank["t"] >= e["t0"] - pre) & (bank["t"] <= e["t0"] + post)].copy()

        if len(win) < pre + post + 1:
            continue

        def predict(d):
            return e[f"{model}_alpha"] + sum(e[f"{model}_{f}"] * d[f] for f in factors)

        win["ar"] = win["exret"] - predict(win)

        est = bank[(bank["t"] >= e["t0"] - 300) & (bank["t"] <= e["t0"] - 50)]
        resid = est["exret"] - predict(est)

        rows.append({
            "ds_name": e["ds_name"],
            "trade_date": e["trade_date"],
            "alpha": e[f"{model}_alpha"],
            "beta": e[f"{model}_mkt_rf"],
            "car": win["ar"].sum(),
            "ar0": win["ar"].iloc[pre], 
            "n_big": int(win["big"].sum()),
            "ar_path": win["ar"].to_numpy(),
            "sigma": resid.std(ddof=len(factors) + 1),     # residual volatility, for Boehmer
            "pos_share": (resid > 0).mean(),               # share of positive ARs, for Cowan
            "r2": 1 - resid.var() / est["exret"].var(),
            "sign": e["sign"],
            "uids": e["uids"],
            "n_other": int(((campaign_t[e["ds_name"]] >= e["t0"] - pre) & (campaign_t[e["ds_name"]] <= e["t0"] + post)
                            & (campaign_t[e["ds_name"]] != e["t0"])).sum()),   # other campaigns inside the window
        })

    return pd.DataFrame(rows)

In [ ]:
studies = {}
for model in ["capm", "ff"]:
    for k in [1, 5, 10]:
        studies[(model, k)] = run_event_study(ev, panel, betas, pre=k, post=k, model=model)

summary = pd.DataFrame({key: s["car"].describe() for key, s in studies.items()}).T
print(summary[["count", "mean", "50%", "std", "min", "max"]].round(4))
print(f"Correlation CAPM / FF on [-10,+10]: {studies[('capm', 10)]['car'].corr(studies[('ff', 10)]['car']):.3f}")
print(studies[("capm", 10)][["sigma", "pos_share"]].describe().round(4))

In [ ]:
# diagnostics of the market model on the estimation window, CAPM only

from statsmodels.stats.diagnostic import het_breuschpagan

data = ev.merge(betas[["ds_name", "t", "capm_alpha", "capm_mkt_rf"]], left_on=["ds_name", "t0"], right_on=["ds_name", "t"], how="left")

rows = []
for _, e in data.iterrows():
    bank = panel[panel["ds_name"] == e["ds_name"]]
    est = bank[(bank["t"] >= e["t0"] - 300) & (bank["t"] <= e["t0"] - 50)]

    x, y = est["mkt_rf"], est["exret"]
    resid = y - (e["capm_alpha"] + e["capm_mkt_rf"] * x)

    half = len(est) // 2
    x1, y1 = x.iloc[:half], y.iloc[:half]
    x2, y2 = x.iloc[half:], y.iloc[half:]

    rows.append({
        "ds_name": e["ds_name"],
        "trade_date": e["trade_date"],
        "dw": (resid.diff() ** 2).sum() / (resid ** 2).sum(),
        "bp_p": het_breuschpagan(resid, sm.add_constant(x))[1],
        "beta_1": x1.cov(y1) / x1.var(),
        "beta_2": x2.cov(y2) / x2.var(),
    })

cars = studies[("capm", 10)].merge(pd.DataFrame(rows), on=["ds_name", "trade_date"])

In [ ]:
print(f"{len(cars):,} events with a CAR")
print(cars[["beta", "car", "ar0"]].describe().round(4))

In [ ]:
low_dw = cars["dw"] < 1.5
cars["beta_diff"] = cars["beta_2"] - cars["beta_1"]
print(f"{low_dw.mean():.1%} of events have a Durbin-Watson < 1.5 (autocorrelated residuals)")
bp_events = cars["bp_p"] < 0.05
print(f"{bp_events.mean():.1%} of events have a Breusch-Pagan p-value < 0.05 (heteroskedastic residuals)")
print(cars[["r2", "dw", "beta_diff"]].describe().round(4))
print(f"Correlation between first- and second-half betas: {cars['beta_1'].corr(cars['beta_2']):.2f}")
print(cars.groupby("ds_name")["r2"].median().nsmallest(6).round(3))

**Diagnostics of the market model:** The market factor explains a median 43% of daily bank returns (R²), which is typical for bank stocks; the lowest R² belong mostly to the illiquid banks with low betas (Swiss National Bank, Umweltbank, the two cantonal banks, Van Lanschot), together with Permanent TSB, an Irish bank rescued by the state. Residuals show no first-order autocorrelation (mean Durbin-Watson 2.01; only 0.4% of events below 1.5). The Breusch-Pagan test rejects homoskedasticity for 15% of events (against 5% expected by chance): this does not bias α and β, but it means that the tests in Part D should not rely on a constant variance. Betas estimated on the two halves of the estimation window show no systematic drift (mean difference -0.03) but vary substantially across the two halves (standard deviation 0.37, correlation 0.62), consistent with changing risk during the euro crisis. We therefore check in Part D that our results hold with a market-adjusted model (β = 1), which does not require estimating β.

In [ ]:
low = cars[cars["beta"] < 0.3]
print(f"{len(low)} events with beta < 0.3, from {low['ds_name'].nunique()} banks:")
print(low["ds_name"].value_counts())
print(zero.reindex(low["ds_name"].unique()).mul(100).round(1).rename("% zero-return days"))

**Result:** Estimated betas are plausible for banks (mean 1.23, median 1.25). Only 14 events (6 banks) have a beta below 0.3: small, domestically focused banks, namely two Swiss cantonal banks, the Swiss National Bank, Van Lanschot Kempen, Umweltbank and SpareBank 1 SR-Bank. Two reasons explain their low betas. First, they trade infrequently: 10–23% of their trading days have a zero return, against a sample median of about 4–5%, which biases betas towards zero. Second, a Europe-wide market factor is a poor benchmark for Swiss and Norwegian banks, which trade in their own currency and market.

In [ ]:
print(f"{(cars['car'] < -0.5).sum()} events with CAR < -50%, {(cars['car'] > 0.3).sum()} with CAR > +30%")
print(cars.nsmallest(5, "car")[["ds_name", "trade_date", "car", "beta"]].round({"car": 3, "beta": 3}))
print(cars.nlargest(5, "car")[["ds_name", "trade_date", "car", "beta"]].round({"car": 3, "beta": 3}))

**Extreme CARs:** A handful of events have very large CARs. In each case a major bank-specific crisis falls inside the event window, so these CARs are most likely driven by that crisis rather than by the NGO campaign:

- **Bankia, May 2013 (CAR ≈ −140%):** the state recapitalisation by the FROB almost wiped out existing shareholders and forcibly converted retail hybrid debt into shares; when trading resumed, these new shareholders sold massively. A CAR below −100% is mechanically possible because daily ARs are summed, not compounded.
- **Monte dei Paschi, December 2016 (−31%):** the private recapitalisation failed, trading was suspended and the Italian state stepped in.
- **Bankinter (May 2012) and Popular (September 2011 and 2013), −25% to −29%:** Spanish banks during or just after the euro-area sovereign and banking crisis.
- **Positive extremes, +31% to +41%:** rebounds after periods of stress, e.g. Deutsche Bank in October 2016, recovering from fears of a $14bn fine from the US Department of Justice, and Bankia in January 2014 after its restructuring.

We draw two lessons. First, a few events dominate the mean, so in Part D we use CARs winsorised at 1%/99% and also report medians. Only one event contains a daily move above 50%, so the extremes do not come from obvious data errors but from genuine crises. Second, most extremes are Spanish and Italian banks hit by national crises, which a Europe-wide market factor cannot capture.

In [ ]:
def winsorise(s, col="car"):
    lo, hi = s[col].quantile([0.01, 0.99])
    return s.assign(car_w=s[col].clip(lo, hi))

In [ ]:
cars = winsorise(cars)
print(f"CARs winsorised at 1%/99%: [{cars['car_w'].min():.1%}, {cars['car_w'].max():.1%}]")
print(f"{(cars['n_big'] > 0).sum()} events with a >50% daily move in the event window")

In [ ]:
cars = cars.merge(
    ev[["ds_name", "trade_date", "sentiment", "negative", "prominence",
        "ngo_power", "n_targets", "climate", "mass", "region"]],
    on=["ds_name", "trade_date"], how="left",
)
print(cars.shape)
cars["ds_name"].nunique()

In [ ]:
# other windows and models, then the handoff file (one row per event, without the daily path)
extra = {"car_1": ("capm", 1), "car_5": ("capm", 5),
         "car_ff": ("ff", 10), "car_ff_1": ("ff", 1), "car_ff_5": ("ff", 5)}

for name, key in extra.items():
    cars = cars.merge(studies[key][["ds_name", "trade_date", "car"]].rename(columns={"car": name}),
                      on=["ds_name", "trade_date"], how="left")

cars.drop(columns="ar_path").to_csv(OUT / "cars.csv", index=False)
print(cars.shape, cars["ds_name"].nunique())

**Output of Part C:** `cars` holds one row per event (1,407 events, 68 banks) with the estimated α and β, the CAR over [t0−10, t0+10] (raw `car` and winsorised `car_w`, bounds [−19.7%, +21.3%]), the event-day abnormal return `ar0`, the 21 daily ARs (`ar_path`), and the campaign characteristics (sentiment, prominence, NGO reach, number of targets, climate topic, region). Part D tests whether these CARs differ from zero and how they vary with the campaign characteristics.

In [ ]:
def clustered_t(y, groups):
    # regressing y on a constant gives its mean; the cluster SE allow correlation within a group
    fit = sm.OLS(y.to_numpy(), np.ones(len(y))).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(groups)[0]})
    return fit.tvalues[0]

def car_tests(s, pre, post, col="car_w"):
    y, n = s[col], len(s)

    scar = y / (s["sigma"] * np.sqrt(pre + post + 1))       # CAR in units of its expected volatility
    p = s["pos_share"].mean()                               # share of positive ARs in normal times

    return pd.Series({
        "n": n,
        "mean": y.mean(),
        "median": y.median(),
        "t_plain": y.mean() / (y.std() / np.sqrt(n)),
        "t_boehmer": scar.mean() * np.sqrt(n) / scar.std(),
        "z_cowan": ((y > 0).sum() - n * p) / np.sqrt(n * p * (1 - p)),
        "t_date": clustered_t(y, s["trade_date"]),
        "t_month": clustered_t(y, s["trade_date"].dt.to_period("M")),
    })

In [ ]:
def test_table(studies, keep=None, col="car_w"):
    rows = {}
    for (model, k), st in studies.items():
        w = winsorise(st)
        if keep is not None:
            w = w[keep(w)]
        for sign in ["neg", "pos"]:
            rows[(model, k, sign)] = car_tests(w[w["sign"] == sign], k, k, col)
        out = pd.DataFrame(rows).T
    out.index.names = ["model", "window", "sign"]
    return out


results = test_table(studies)
clean = test_table(studies, keep=lambda w: w["n_other"] == 0)

print(pd.Series({k: (s["n_other"] > 0).mean() for k, s in studies.items()}).round(3))
print(clean.round(3).to_string())

In [ ]:
# market-adjusted model: alpha = 0 and beta = 1
MODELS["mkt"] = ["mkt_rf"]
betas["mkt_alpha"], betas["mkt_mkt_rf"] = 0.0, 1.0
for k in [1, 5, 10]:
    studies[("mkt", k)] = run_event_study(ev, panel, betas, pre=k, post=k, model="mkt")

stale = zero[zero > 0.10].index
print(f"{len(stale)} banks with more than 10% zero-return days: {list(stale)}")

checks = {
    "baseline": test_table(studies),
    "clean": test_table(studies, keep=lambda w: w["n_other"] == 0),
    "raw": test_table(studies, col="car"),
    "no_stale": test_table(studies, keep=lambda w: ~w["ds_name"].isin(stale)),
}

print((pd.DataFrame({name: t["mean"] for name, t in checks.items()}) * 100).round(2).to_string())
print(pd.DataFrame({name: t["t_month"] for name, t in checks.items()}).round(2).to_string())

s = studies[("capm", 10)]
print(s.assign(dirty=s["n_other"] > 0).groupby("ds_name")["dirty"].agg(["size", "mean"])
      .sort_values("size", ascending=False).head(8).round(2))

In [ ]:
# data check: bank-years where more than half of the days have a zero return
p = panel[panel["date"] >= "2010-01-01"].assign(zero=lambda d: d["ret_local"].abs() < 1e-12)
by_year = p.groupby(["ds_name", p["date"].dt.year])["zero"].mean()
print(by_year[by_year > 0.5].round(2).to_string())

In [ ]:
def build_ev(gap):
    e = events[overlap_rule(events, gap)].merge(panel[["ds_name", "date", "t"]], left_on=["ds_name", "trade_date"],
                                                right_on=["ds_name", "date"], how="inner", suffixes=("", "_panel"))
    e = e.rename(columns={"t": "t0"})
    return e[e["t0"] >= 300].copy()


rows = {}
for gap in [30, 45, 60, 90]:
    e = build_ev(gap)
    for model in ["capm", "ff"]:
        w = winsorise(run_event_study(e, panel, betas, model=model))
        for sign in ["neg", "pos"]:
            rows[(gap, model, sign)] = car_tests(w[w["sign"] == sign], 10, 10)

gaps = pd.DataFrame(rows).T
gaps.index.names = ["gap", "model", "sign"]
print(gaps[["n", "mean", "median", "t_boehmer", "t_month"]].round(4).to_string())